Code Adapted from
https://github.com/handsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb

# Packages, Google Drive, and Reading Data

In [ ]:
import json, random
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sentence_transformers import SentenceTransformer

drive.mount('/content/drive', force_remount=True)
path = "/content/drive/MyDrive/DSA 495 Project/"
file = f"{path}Data/Video_Games.jsonl"


Mounted at /content/drive


# Sampling & Splitting

In [ ]:
# sample
random.seed(495)
rows = []
with open(file, 'r') as fp:
    for line in fp:
        if random.random() < 0.005:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
df = df[["rating", "title", "text"]]

# clean + combine
def make_text(d):
    body = d["text"].fillna("").str.replace(r"<br\s*/?>", " ", regex=True)
    return (d["title"].fillna("") + ". " + body).str.replace(r"\s+", " ", regex=True).str.strip()

df["review"] = make_text(df)

# split
train_df, valid_test_df = train_test_split(df, test_size=0.3, random_state=42)
valid_df, test_df = train_test_split(valid_test_df, test_size=0.5, random_state=42)

# Embeddings & Ridge Regression
Compared to OLS, ridge regression shrinks the weight, using L2-Norm, on patterns that barely vary across reviews, because those are where OLS estimates are least stable. Patterns that vary a lot across reviews are mostly left alone, so they carry relatively more weight in the model.

In [ ]:
# embed
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
X_train = model.encode(train_df["review"].tolist(), batch_size=64, show_progress_bar=True)
X_valid = model.encode(valid_df["review"].tolist(), batch_size=64, show_progress_bar=True)
X_test  = model.encode(test_df["review"].tolist(),  batch_size=64, show_progress_bar=True)

y_train, y_valid, y_test = train_df["rating"].values, valid_df["rating"].values, test_df["rating"].values

# baseline + alpha sweep
print("Mean baseline MSE:", mean_squared_error(y_valid, np.full_like(y_valid, y_train.mean())))
for a in [0.1, 1, 10, 100]:
    pred = np.clip(Ridge(alpha=a).fit(X_train, y_train).predict(X_valid), 1, 5)
    print(a, mean_squared_error(y_valid, pred))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (7 outputs left out)

Batches:   0%|          | 0/253 [00:00<?, ?it/s]

Batches:   0%|          | 0/55 [00:00<?, ?it/s]

Batches:   0%|          | 0/55 [00:00<?, ?it/s]

Mean baseline MSE: 2.1119432502192805
0.1 0.6468407270243975
1 0.6593058874396864
10 0.7396745387455853
100 1.0073330609774949
